In [1]:
import os
from dotenv import load_dotenv

load_dotenv()
cwd = os.environ["REPO_ROOT"]
print(cwd)
os.chdir(cwd)

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from utils.download_data import download_data
import math


pd.set_option('display.max_colwidth', 50)
pd.set_option('display.max_columns', None)

/home/ruchirich/Documents/repositories/toronto-open-data/tpl


# Load Data

read csv and impute values

## 0. Branch Info

In [14]:
# Datasets are called "packages". Each package can contain many "resources"
# To retrieve the metadata for this package and its resources, use the package name in this page's URL:
params = { "id": "library-branch-general-information"}
branch_info_file = cwd+"/data/library-branch-general-information.csv"

download_data(params, branch_info_file)
# usecols = ["BranchCode", "BranchName", "PhysicalBranch"]
# branch_info = pd.read_csv(branch_info_file, usecols=usecols)
branch_info = pd.read_csv(branch_info_file)

## 1. Circulation

In [4]:
# Datasets are called "packages". Each package can contain many "resources"
# To retrieve the metadata for this package and its resources, use the package name in this page's URL:
params = { "id": "library-circulation"}
circulation_file = cwd+"/data/library-circulation.csv"

download_data(params, circulation_file)
usecols = ['Year', 'BranchCode', 'Circulation']
circulation = pd.read_csv(circulation_file, usecols=usecols, parse_dates=["Year"], date_format="%Y")
circulation['Year'] = circulation["Year"].dt.year.astype("object")
# pivot
circulation_by_branch = circulation.pivot(columns='BranchCode',index='Year',values='Circulation')

# impute
circulation_by_branch_imputed = circulation_by_branch.copy(deep=True)
imputation_strategies = {"median": ["ME"],
                        "constant": ["FO", "SC"]}

for strategy, columns in imputation_strategies.items():
    cols_to_impute = [col for col in columns if col in circulation_by_branch_imputed.columns]

    if strategy == "constant":
        imputer = SimpleImputer(missing_values=pd.NA, strategy=strategy, fill_value=0)
    else:
        imputer = SimpleImputer(missing_values=pd.NA, strategy=strategy)
    
    circulation_by_branch_imputed[cols_to_impute] = imputer.fit_transform(circulation_by_branch_imputed[cols_to_impute])

# melt back to long format
circulation = circulation_by_branch_imputed.reset_index().melt(id_vars="Year", value_name="Circulation", var_name="BranchCode")

## 2. Visits

In [5]:
# Datasets are called "packages". Each package can contain many "resources"
# To retrieve the metadata for this package and its resources, use the package name in this page's URL:
params = { "id": "library-visits"}
visits_file = cwd+"/data/library-visits.csv"

download_data(params, visits_file)
usecols = ['Year', 'BranchCode', 'Visits']
visits = pd.read_csv(visits_file, usecols=usecols, parse_dates=["Year"], date_format="%Y")
visits['Year'] = visits["Year"].dt.year.astype("object")
# pivot
visits_by_branch = visits.pivot(columns='BranchCode',index='Year',values='Visits')

# impute
visits_by_branch_imputed = visits_by_branch.copy(deep=True)
imputation_strategies = {"median": ["WY", "MP", "MD", "CL", "FV", "SI"],
                        "constant": ["FO", "SC", "CH", "SW"]}

for strategy, columns in imputation_strategies.items():
    cols_to_impute = [col for col in columns if col in visits_by_branch_imputed.columns]

    if strategy == "constant":
        imputer = SimpleImputer(missing_values=np.nan, strategy=strategy, fill_value=0)
    else:
        imputer = SimpleImputer(missing_values=np.nan, strategy=strategy)
    
    visits_by_branch_imputed[cols_to_impute] = imputer.fit_transform(visits_by_branch_imputed[cols_to_impute])

# melt back to long format
visits = visits_by_branch_imputed.reset_index().melt(id_vars="Year", value_name="Visits", var_name="BranchCode")

## 3. Registrations

In [7]:
# Datasets are called "packages". Each package can contain many "resources"
# To retrieve the metadata for this package and its resources, use the package name in this page's URL:
# params = { "id": "library-registrations"}
# registrations_file = cwd+"/data/library-registrations.csv"

# download_data(params, registrations_file)
usecols = ['Year', 'BranchCode', 'Registrations']
registrations = pd.read_csv("data/library-card-registrations.csv", usecols=usecols, parse_dates=["Year"], date_format="%Y")
registrations['Year'] = registrations["Year"].dt.year.astype("object")
# pivot
registrations_by_branch = registrations.pivot(columns='BranchCode',index='Year',values='Registrations')

# impute
registrations_by_branch_imputed = registrations_by_branch.copy(deep=True)
imputation_strategies = {"median": ["BKTWO"],
                        "most_frequent": ["AL", "LD", "ME", "OS"],
                        "constant": ["VIR", "FO", "SC"]}

for strategy, columns in imputation_strategies.items():
    cols_to_impute = [col for col in columns if col in registrations_by_branch_imputed.columns]

    if strategy == "constant":
        imputer = SimpleImputer(missing_values=np.nan, strategy=strategy, fill_value=0)
    else:
        imputer = SimpleImputer(missing_values=np.nan, strategy=strategy)
    
    registrations_by_branch_imputed[cols_to_impute] = imputer.fit_transform(registrations_by_branch_imputed[cols_to_impute])

# melt back to long format
registrations = registrations_by_branch_imputed.reset_index().melt(id_vars="Year", value_name="Registrations", var_name="BranchCode")

## 4. Sessions

In [11]:
# Datasets are called "packages". Each package can contain many "resources"
# To retrieve the metadata for this package and its resources, use the package name in this page's URL:
# params = { "id": "library-sessions"}
# sessions_file = cwd+"/data/library-sessions.csv"

# download_data(params, sessions_file)
usecols = ['Year', 'BranchCode', 'Sessions']
sessions = pd.read_csv("data/library-workstation-usage.csv", usecols=usecols, parse_dates=["Year"], date_format="%Y")
sessions['Year'] = sessions["Year"].dt.year.astype("object")
# pivot
sessions_by_branch = sessions.pivot(columns='BranchCode',index='Year',values='Sessions')

# impute
sessions_by_branch_imputed = sessions_by_branch.copy(deep=True)
imputation_strategies = {"median": ["CL", "FV", "MD"],
                        "constant": ["FO", "SC"]}

for strategy, columns in imputation_strategies.items():
    cols_to_impute = [col for col in columns if col in sessions_by_branch_imputed.columns]

    if strategy == "constant":
        imputer = SimpleImputer(missing_values=np.nan, strategy=strategy, fill_value=0)
    else:
        imputer = SimpleImputer(missing_values=np.nan, strategy=strategy)
    
    sessions_by_branch_imputed[cols_to_impute] = imputer.fit_transform(sessions_by_branch_imputed[cols_to_impute])

# melt back to long format
sessions = sessions_by_branch_imputed.reset_index().melt(id_vars="Year", value_name="Sessions", var_name="BranchCode")

## 5. Events

In [ ]:
# Datasets are called "packages". Each package can contain many "resources"
# To retrieve the metadata for this package and its resources, use the package name in this page's URL:
params = { "id": "library-branch-programs-and-events-feed"}
branch_info_file = cwd+"/data/library-branch-programs-and-events-feed.csv"

download_data(params, branch_info_file)
events = pd.read_csv(branch_info_file)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4401 entries, 0 to 4400
Data columns (total 26 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   _id              4401 non-null   int64  
 1   title            4401 non-null   object 
 2   startdate        4401 non-null   object 
 3   enddate          24 non-null     object 
 4   starttime        4377 non-null   object 
 5   endtime          4377 non-null   object 
 6   library          4401 non-null   object 
 7   location         2200 non-null   object 
 8   description      4401 non-null   object 
 9   pagelink         4401 non-null   object 
 10  id               4401 non-null   int64  
 11  rcid             4401 non-null   int64  
 12  eventtype1       4401 non-null   object 
 13  eventtype2       1572 non-null   object 
 14  eventtype3       233 non-null    object 
 15  agegroup1        4401 non-null   object 
 16  agegroup2        1245 non-null   object 
 17  agegroup3     

# Merge Data

In [15]:
# Merge dataframes
usage_data = (circulation.merge(visits, on=["Year", "BranchCode"])
                .merge(registrations, on=["Year", "BranchCode"])
                .merge(sessions, how="left", on=["Year", "BranchCode"]))

full_data = usage_data.merge(branch_info, on="BranchCode")

# Clean event descriptions
events['description'] = events['description'].str.replace(r'<[^<>]*>', '', regex=True)

In [19]:
full_data.to_csv(path_or_buf="data/merged_data.csv", index=False)
events.to_csv(path_or_buf="data/cleaned_events.csv", index=False)